In [ ]:
import json
import subprocess
import time
import requests
import os
import re

try:
    OPENROUTER_API_KEY = os.environ['OPENROUTER_API_KEY']
    #print(f"API Key: {OPENROUTER_API_KEY}")
except KeyError:
    print("OPENROUTER_API_KEY environment variable not set.")

def generate_code(description):
    prompt = f"""You are a competitive programming assistant.
                Write a complete, efficient C++ program using standard input/output to solve this problem 
                Please do not include explanation. Only Code:

                {description}
            """   
    
    try:
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json",
                "HTTP-Referer": "https://openrouter.ai",
            },
            json={  
                "model": "qwen/qwen-2.5-coder-32b-instruct:free", #"model": "x-ai/grok-code-fast-1",
                "messages": [{"role": "user", "content": prompt}],
            },
            timeout=120,
        )
        
        #return response # fortesting the format of the response
        
    except requests.RequestException as e:
        print("Network error:", e)
        return None
    
    print("Status:", response.status_code)
    if response.status_code != 200:
        print("Response text:", response.text[:500])
        return None


    content = response.json()["choices"][0]["message"]["content"]
    
    cpp_code = re.search(r"```cpp\s*(.*?)```", content, re.DOTALL)
    if cpp_code:
        return cpp_code.group(1).strip()

    return cpp_code

In [28]:
#desc = """You are given an array of n positive integers a_1, a_2, ..., a_n. You can perform the following operation any number of times: select several distinct indices i_1, i_2, ..., i_k (1 ≤ i_j ≤ n) and move the number standing at the position i_1 to the position i_2, the number at the position i_2 to the position i_3, ..., the number at the position i_k to the position i_1. In other words, the operation cyclically shifts elements: i_1 → i_2 → … i_k → i_1.\n\nFor example, if you have n=4, an array a_1=10, a_2=20, a_3=30, a_4=40, and you choose three indices i_1=2, i_2=1, i_3=4, then the resulting array would become a_1=20, a_2=40, a_3=30, a_4=10.\n\nYour goal is to make the array sorted in non-decreasing order with the minimum number of operations. The additional constraint is that the sum of cycle lengths over all operations should be less than or equal to a number s. If it's impossible to sort the array while satisfying that constraint, your solution should report that as well.\n\nInput\n\nThe first line of the input contains two integers n and s (1 ≤ n ≤ 200 000, 0 ≤ s ≤ 200 000)—the number of elements in the array and the upper bound on the sum of cycle lengths.\n\nThe next line contains n integers a_1, a_2, ..., a_n—elements of the array (1 ≤ a_i ≤ 10^9).\n\nOutput\n\nIf it's impossible to sort the array using cycles of total length not exceeding s, print a single number "-1" (quotes for clarity).\n\nOtherwise, print a single number q— the minimum number of operations required to sort the array.\n\nOn the next 2 ⋅ q lines print descriptions of operations in the order they are applied to the array. The description of i-th operation begins with a single line containing one integer k (1 ≤ k ≤ n)—the length of the cycle (that is, the number of selected indices). The next line should contain k distinct integers i_1, i_2, ..., i_k (1 ≤ i_j ≤ n)—the indices of the cycle.\n\nThe sum of lengths of these cycles should be less than or equal to s, and the array should be sorted after applying these q operations.\n\nIf there are several possible answers with the optimal q, print any of them.\n\nExamples\n\nInput\n\n5 5\n3 2 3 1 1\n\n\nOutput\n\n1\n5\n1 4 2 3 5 \n\n\nInput\n\n4 3\n2 1 4 3\n\n\nOutput\n\n-1\n\nInput\n\n2 0\n2 2\n\n\nOutput\n\n0\n\nNote\n\nIn the first example, it's also possible to sort the array with two operations of total length 5: first apply the cycle 1 → 4 → 1 (of length 2), then apply the cycle 2 → 3 → 5 → 2 (of length 3). However, it would be wrong answer as you're asked to use the minimal possible number of operations, which is 1 in that case.\n\nIn the second example, it's possible to the sort the array with two cycles of total length 4 (1 → 2 → 1 and 3 → 4 → 3). However, it's impossible to achieve the same using shorter cycles, which is required by s=3.\n\nIn the third example, the array is already sorted, so no operations are needed. Total length of empty set of cycles is considered to be zero."""
#desc = "write a simple cpp function. no explanation neeed. only code"
#content = generate_code(desc)
print("qwenstart")

qwenstart


In [29]:
def main():
    
    #MODEL_NAME = "codellama:7b-instruct" #
    #INPUT_FILE = "../json_descriptions/descriptions-train-00001.json" 
    #OUTPUT_FILE = "json_aigc_grok/AIGC_train-00001.json" #desc_with_LLM_code_1.json
    
    
    INPUT_FILE = "../json_descriptions/descriptions-train-00002.json" 
    OUTPUT_FILE = "json_aigc_qwen2.5_coder_32b/AIGC_train-00002_3.json" #desc_with_LLM_code_1.json
    
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        descriptions = json.load(f)

    results = []

    for entry in descriptions:
        desc = entry["description"]
        print(f"Generating: {entry.get('name', '')}")

        try:
            cpp_code = generate_code(desc)

            results.append({
                "name": entry.get("name", ""),
                "generated_cpp": cpp_code.strip()
            })

            time.sleep(0.5)

        except Exception as e:
            print(f"Error with {entry.get('name', '')}: {e}")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"Saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Generating: 825_E. Minimal Labels
Status: 200
Generating: 848_C. Goodbye Souvenir
Status: 200
Generating: 870_A. Search for Pretty Integers
Status: 200
Generating: 896_E. Welcome home, Chtholly
Status: 200
Generating: 918_A. Eleven
Status: 200
Generating: 940_A. Points on the line
Status: 200
Generating: 967_D. Resource Distribution
Status: 200
Generating: 993_D. Compute Power
Status: 200
Generating: p02630 AtCoder Beginner Contest 171 - Replacing
Status: 200
Generating: p02761 AtCoder Beginner Contest 157 - Guess The Number
Status: 200
Generating: p02896 AtCoder Grand Contest 039 - Min Product Sum
Status: 200
Generating: p03031 AtCoder Beginner Contest 128 - Switches
Status: 200
Generating: p03172 Educational DP Contest - Candies
Status: 200
Generating: p03318 AtCoder Beginner Contest 101 - Snuke Numbers
Status: 200
Generating: p03474 AtCoder Beginner Contest 084 - Postal Code
Status: 200
Generating: p03637 AtCoder Beginner Contest 069 - 4-adjacent
Status: 200
Generating: p03794 Mujin

Status: 200
Generating: 1342_E. Placing Rooks
Status: 200
Generating: 1364_E. X-OR
Status: 200
Generating: 1384_F. Rearrange
Status: 200
Generating: 1406_C. Link Cut Centroids
Status: 200
Generating: 1427_D. Unshuffling a Deck
Status: 200
Generating: 1450_D. Rating Compression
Status: 200
Generating: 1474_B. Different Divisors
Status: 200
Generating: 149_D. Coloring Brackets
Status: 200
Generating: 1523_D. Love-Hate
Status: 200
Generating: 155_C. Hometask
Status: 200
Generating: 177_D1. Encrypting Messages
Status: 200
Generating: 198_D. Cube Snake
Status: 200
Generating: 221_E. Little Elephant and Shifts
Status: 200
Generating: 245_F. Log Stream Analysis
Status: 200
Generating: 270_D. Greenhouse Effect
Status: 200
Generating: 294_A. Shaass and Oskols
Status: 200
Generating: 317_D. Game with Powers
Status: 200
Generating: 341_D. Iahub and Xors
Status: 200
Generating: 365_A. Good Number
Status: 200
Generating: 388_E. Fox and Meteor Shower
Status: 200
Generating: 409_C. Magnum Opus
Status

Status: 200
Generating: p02797 Keyence Programming Contest 2020 - Subarray Sum
Status: 200
Generating: p02933 AtCoder Beginner Contest 138 - Red or Not
Status: 200
Generating: p03069 Tenka1 Programmer Beginner Contest 2019 - Stones
Status: 200
Generating: p03212 AtCoder Beginner Contest 114 - 755
Status: 200
Generating: p03361 AtCoder Beginner Contest 096 - Grid Repainting 2
Status: 200
Generating: p03519 CODE FESTIVAL 2017 Elimination Tournament Round 1 (Parallel) - Paired Parentheses
Status: 200
Generating: p03684 AtCoder Regular Contest 076 - Built?
Status: 200
Generating: p03837 AtCoder Beginner Contest 051 - Candidates of No Shortest Paths
Status: 200
Generating: p04004 AtCoder Regular Contest 061 - Card Game for Three
Status: 200
Saved to json_aigc_qwen2.5_coder_32b/AIGC_train-00002_3.json


## Meta llama 3.3


In [ ]:
import json
import subprocess
import time
import requests
import os
import re

try:
    OPENROUTER_API_KEY = os.environ['OPENROUTER_API_KEY']
    #print(f"API Key: {OPENROUTER_API_KEY}")
except KeyError:
    print("OPENROUTER_API_KEY environment variable not set.")

def generate_code(description):
    prompt = f"""You are a competitive programming assistant.
                Write a complete, efficient C++ program using standard input/output to solve this problem 
                Please do not include explanation. Only Code:

                {description}
            """   
    
    try:
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json",
                "HTTP-Referer": "https://openrouter.ai",
            },
            json={  
                "model": "meta-llama/llama-3.3-70b-instruct:free", #"model": "x-ai/grok-code-fast-1",
                "messages": [{"role": "user", "content": prompt}],
            },
            timeout=120,
        )
        
        #return response # fortesting the format of the response
        
    except requests.RequestException as e:
        print("Network error:", e)
        return None
    
    print("Status:", response.status_code)
    if response.status_code != 200:
        print("Response text:", response.text[:500])
        return None

    
    content = response.json()["choices"][0]["message"]["content"]
    
    cpp_code = re.search(r"```cpp\s*(.*?)```", content, re.DOTALL)
    if cpp_code:
        return cpp_code.group(1).strip()

    return cpp_code




def main():
    
    print("llama")
    INPUT_FILE = "../json_descriptions/descriptions-train-00002.json" 
    OUTPUT_FILE = "../llama-3.3-70b-instruct/json_aigc_llama-3.3-70b-instruct/AIGC_train-00002_3.json" #desc_with_LLM_code_1.json
    
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        descriptions = json.load(f)

    results = []

    for entry in descriptions:
        desc = entry["description"]
        print(f"Generating: {entry.get('name', '')}")

        try:
            cpp_code = generate_code(desc)

            results.append({
                "name": entry.get("name", ""),
                "generated_cpp": cpp_code.strip()
            })

            time.sleep(0.5)

        except Exception as e:
            print(f"Error with {entry.get('name', '')}: {e}")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"Saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

llama
Generating: 825_E. Minimal Labels
Status: 200
Generating: 848_C. Goodbye Souvenir
Status: 200
Generating: 870_A. Search for Pretty Integers
Status: 200
Generating: 896_E. Welcome home, Chtholly
Status: 200
Generating: 918_A. Eleven
Status: 200
Generating: 940_A. Points on the line
Status: 200
Generating: 967_D. Resource Distribution
Status: 200
Generating: 993_D. Compute Power
Status: 200
Generating: p02630 AtCoder Beginner Contest 171 - Replacing
Status: 200
Generating: p02761 AtCoder Beginner Contest 157 - Guess The Number
Status: 200
Generating: p02896 AtCoder Grand Contest 039 - Min Product Sum
Status: 200
Generating: p03031 AtCoder Beginner Contest 128 - Switches
Status: 200
Generating: p03172 Educational DP Contest - Candies
Status: 200
Generating: p03318 AtCoder Beginner Contest 101 - Snuke Numbers
Status: 200
Generating: p03474 AtCoder Beginner Contest 084 - Postal Code
Status: 200
Generating: p03637 AtCoder Beginner Contest 069 - 4-adjacent
Status: 200
Generating: p03794

Status: 200
Generating: 409_C. Magnum Opus
Status: 200
Generating: 436_D. Pudding Monsters
Status: 200
Generating: 459_E. Pashmak and Graph
Status: 200
Generating: 480_D. Parcels
Status: 200
Generating: 505_D. Mr. Kitayuta's Technology
Status: 200
Generating: 529_C. Rooks and Rectangles
Status: 200
Generating: 554_E. Love Triangles
Status: 200
Generating: 580_D. Kefa and Dishes
Status: 200
Generating: 602_C. The Two Routes
Status: 200
Generating: 625_A. Guest From the Past
Status: 200
Generating: 673_C. Bear and Colors
Status: 200
Generating: 698_D. Limak and Shooting Points
Status: 200
Generating: 719_E. Sasha and Array
Status: 200
Generating: 740_B. Alyona and flowers
Status: 200
Generating: 764_A. Taymyr is calling you
Status: 200
Generating: 787_C. Berzerk
Status: 200
Generating: 808_G. Anthem of Berland
Status: 200
Generating: 833_C. Ever-Hungry Krakozyabra
Status: 200
Generating: 854_A. Fraction
Status: 200
Generating: 878_D. Magic Breeding
Status: 200
Generating: 902_A. Visiting

# gemma3

In [ ]:
import json
import subprocess
import time
import requests
import os
import re

try:
    OPENROUTER_API_KEY = os.environ['OPENROUTER_API_KEY']
    #print(f"API Key: {OPENROUTER_API_KEY}")
except KeyError:
    print("OPENROUTER_API_KEY environment variable not set.")

def generate_code(description):
    prompt = f"""You are a competitive programming assistant.
                Write a complete, efficient C++ program using standard input/output to solve this problem 
                Please do not include explanation. Only Code:

                {description}
            """   
    
    try:
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json",
                "HTTP-Referer": "https://openrouter.ai",
            },
            json={  
                "model": "google/gemma-3-27b-it:free", #"model": "x-ai/grok-code-fast-1",
                "messages": [{"role": "user", "content": prompt}],
            },
            timeout=120,
        )
        
        #return response # fortesting the format of the response
        
    except requests.RequestException as e:
        print("Network error:", e)
        return None
    
    print("Status:", response.status_code)
    if response.status_code != 200:
        print("Response text:", response.text[:500])
        return None
  
    content = response.json()["choices"][0]["message"]["content"]
    
    cpp_code = re.search(r"```cpp\s*(.*?)```", content, re.DOTALL)
    if cpp_code:
        return cpp_code.group(1).strip()

    return cpp_code



def main():
    
    print("gemma")
    
    
    INPUT_FILE = "../json_descriptions/descriptions-train-00002.json" 
    OUTPUT_FILE = "../gemma-3-27b-it/json_aigc_gemma-3-27b-it/AIGC_train-00002_3.json" #desc_with_LLM_code_1.json
    
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        descriptions = json.load(f)

    results = []

    for entry in descriptions:
        desc = entry["description"]
        print(f"Generating: {entry.get('name', '')}")

        try:
            cpp_code = generate_code(desc)

            results.append({
                "name": entry.get("name", ""),
                "generated_cpp": cpp_code.strip()
            })

            time.sleep(.5)

        except Exception as e:
            print(f"Error with {entry.get('name', '')}: {e}")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"Saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

gemma
Generating: 825_E. Minimal Labels
Status: 200
Generating: 848_C. Goodbye Souvenir
Status: 200
Generating: 870_A. Search for Pretty Integers
Status: 200
Generating: 896_E. Welcome home, Chtholly
Status: 200
Generating: 918_A. Eleven
Status: 200
Generating: 940_A. Points on the line
Status: 200
Generating: 967_D. Resource Distribution
Status: 200
Generating: 993_D. Compute Power
Status: 200
Generating: p02630 AtCoder Beginner Contest 171 - Replacing
Status: 200
Generating: p02761 AtCoder Beginner Contest 157 - Guess The Number
Status: 200
Generating: p02896 AtCoder Grand Contest 039 - Min Product Sum
Status: 200
Generating: p03031 AtCoder Beginner Contest 128 - Switches
Status: 200
Generating: p03172 Educational DP Contest - Candies
Status: 200
Generating: p03318 AtCoder Beginner Contest 101 - Snuke Numbers
Status: 200
Generating: p03474 AtCoder Beginner Contest 084 - Postal Code
Status: 200
Generating: p03637 AtCoder Beginner Contest 069 - 4-adjacent
Status: 200
Generating: p03794

Status: 200
Generating: 1215_B. The Number of Products
Status: 200
Generating: 1238_D. AB-string
Status: 200
Error with 1238_D. AB-string: 'NoneType' object has no attribute 'strip'
Generating: 1256_E. Yet Another Division Into Teams
Status: 200
Generating: 127_E. E-reader Display
Status: 200
Generating: 12_D. Ball
Status: 200
Generating: 1323_F. Reality Show
Status: 200
Generating: 1342_E. Placing Rooks
Status: 200
Generating: 1364_E. X-OR
Status: 200
Generating: 1384_F. Rearrange
Status: 200
Error with 1384_F. Rearrange: 'NoneType' object has no attribute 'strip'
Generating: 1406_C. Link Cut Centroids
Status: 200
Generating: 1427_D. Unshuffling a Deck
Status: 200
Generating: 1450_D. Rating Compression
Status: 200
Generating: 1474_B. Different Divisors
Status: 200
Generating: 149_D. Coloring Brackets
Status: 200
Generating: 1523_D. Love-Hate
Status: 200
Error with 1523_D. Love-Hate: 'choices'
Generating: 155_C. Hometask
Status: 200
Error with 155_C. Hometask: 'choices'
Generating: 177

Generating: 999_F. Cards and Joy
Status: 200
Generating: p02537 ACL Beginner Contest - Flat Subsequence
Status: 200
Generating: p02668 NOMURA Programming Competition 2020 - Sorting Game
Status: 200
Generating: p02797 Keyence Programming Contest 2020 - Subarray Sum
Status: 200
Generating: p02933 AtCoder Beginner Contest 138 - Red or Not
Status: 200
Generating: p03069 Tenka1 Programmer Beginner Contest 2019 - Stones
Status: 200
Generating: p03212 AtCoder Beginner Contest 114 - 755
Status: 200
Generating: p03361 AtCoder Beginner Contest 096 - Grid Repainting 2
Status: 200
Generating: p03519 CODE FESTIVAL 2017 Elimination Tournament Round 1 (Parallel) - Paired Parentheses
Status: 200
Generating: p03684 AtCoder Regular Contest 076 - Built?
Status: 200
Error with p03684 AtCoder Regular Contest 076 - Built?: 'NoneType' object has no attribute 'strip'
Generating: p03837 AtCoder Beginner Contest 051 - Candidates of No Shortest Paths
Status: 200
Generating: p04004 AtCoder Regular Contest 061 - C